# 11. OpenTelemetry Export

Distributed tracing for ArcLLM: every `invoke()` becomes a span, every module
contributes child spans, and the GenAI semantic conventions make the resulting
trees auto-render in Jaeger, Datadog, Grafana Tempo, and friends.

This notebook is **mock-first** — it captures spans into an
`InMemorySpanExporter`, so it runs end-to-end without an OTLP collector and
without an API key. The OTLP code paths are demonstrated by *config*, not by
making real network calls.

You will learn:

- How `OtelModule` and `BaseModule._span()` together form a two-layer
  instrumentation strategy.
- The exact shape of the span tree: parent (`arcllm.invoke`) → child modules
  (`security`, `retry.attempt.N`, etc.).
- Every standard `gen_ai.*` attribute set on the root span.
- The **new** `arc.queue.*` attributes contributed by `QueueModule` (v0.4):
  `wait_ms`, `depth`, `call_timeout_ms`, `rejected`.
- How to wire `InMemorySpanExporter` for tests and OTLP for production.
- How `clear_cache()` resets the SDK setup flag (`reset_sdk()`) so each
  notebook / test starts from a clean tracer-provider state.


## 1. Setup

The canonical Arc walkthrough boilerplate plus a tiny tracer-provider helper that we'll use throughout to capture spans into memory.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Now we wire up an `InMemorySpanExporter`. Every span produced by ArcLLM in
this notebook will land in `EXPORTER.get_finished_spans()` once the span ends.

We use `SimpleSpanProcessor` (synchronous) instead of `BatchSpanProcessor` so
that spans are flushed immediately — much easier to reason about in a notebook
than waiting for a 5s batch interval.

In [ ]:
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter,
)


def install_in_memory_provider() -> InMemorySpanExporter:
    """Replace the global tracer provider with one that exports to memory."""
    exporter = InMemorySpanExporter()
    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(exporter))
    otel_trace.set_tracer_provider(provider)
    return exporter


EXPORTER = install_in_memory_provider()
print("InMemorySpanExporter installed:", EXPORTER)

A small helper to print spans in a tree shape. It's purely cosmetic — the
real point is that span IDs/parent IDs let any backend reconstruct the full
trace.

In [ ]:
from collections import defaultdict


def print_tree(spans) -> None:
    by_parent: dict[int | None, list] = defaultdict(list)
    for s in spans:
        parent_id = s.parent.span_id if s.parent else None
        by_parent[parent_id].append(s)

    def walk(parent_id: int | None, depth: int) -> None:
        for s in sorted(by_parent[parent_id], key=lambda x: x.start_time):
            print(f"{'  ' * depth}- {s.name}")
            walk(s.context.span_id, depth + 1)

    walk(None, 0)


def reset_exporter() -> None:
    EXPORTER.clear()

## 2. OTel basics in arcllm

OTel integration in arcllm is split across **two layers** by design.

| Layer | What it does | Where |
|-------|--------------|-------|
| `OtelModule` | Creates the *root* `arcllm.invoke` span and tags it with GenAI attributes; optionally configures the SDK | `arcllm/modules/otel.py` |
| `BaseModule._span()` | Context manager every module uses to create *child* spans | `arcllm/modules/base.py` |

Crucially, `opentelemetry-api` is a hard dependency (~100 KB), but
`opentelemetry-sdk` is optional (`pip install arcllm[otel]`). When the SDK
isn't configured, every span is a `NonRecordingSpan` — zero overhead, zero
errors. The same code paths work whether or not exporters exist.

In [ ]:
from arcllm.modules.base import BaseModule
from arcllm.modules.otel import OtelModule, reset_sdk

print("BaseModule:", BaseModule.__module__, BaseModule.__name__)
print("OtelModule:", OtelModule.__module__, OtelModule.__name__)
print("reset_sdk():", reset_sdk.__doc__.splitlines()[0])

`BaseModule._tracer` returns `trace.get_tracer("arcllm")` so every module in
the stack shares one logical tracer. `_span()` is a context manager that:

1. Calls `start_as_current_span(name)` so child spans auto-nest under any
   active parent (including parents from agent frameworks like LangGraph).
2. On exception, records the exception event and sets `StatusCode.ERROR`.
3. Re-raises — never swallows.

The point: a custom module subclass needs **zero OTel boilerplate** to
participate in distributed traces. Just `with self._span("my.module"):`.

In [ ]:
import contextlib
import inspect

# Confirm _span is a context manager and walk its source briefly.
print(
    "contextmanager:",
    isinstance(BaseModule.__dict__["_span"], type(contextlib.contextmanager(lambda: (yield)))),
)
print()
src = inspect.getsource(BaseModule._span)
print(src.strip())

## 3. Span hierarchy across the module stack

The default v0.4 stack order is:

```
Otel  →  Queue  →  Telemetry  →  CircuitBreaker  →  Audit  →  Security  →  Retry  →  Fallback  →  RateLimit  →  Adapter
```

OTel sits **outermost**. That means the root `arcllm.invoke` span is created
first and contains everything else. Modules deeper in the stack create child
spans inside that context. No module has to *know* the OTel parent — context
propagation handles it.

Let's confirm the shape with a tiny mock provider, the `OtelModule`, and a
hand-built parent context.

In [ ]:
from unittest.mock import AsyncMock, MagicMock

from arcllm.types import LLMProvider, LLMResponse, Message, Usage


def make_inner(name: str = "anthropic", model: str = "claude-sonnet-4-6") -> LLMProvider:
    inner = MagicMock(spec=LLMProvider)
    inner.name = name
    inner.model_name = model
    inner.invoke = AsyncMock(
        return_value=LLMResponse(
            content="Hello from mock",
            usage=Usage(input_tokens=42, output_tokens=7, total_tokens=49),
            model=model,
            stop_reason="end_turn",
        )
    )
    return inner


inner = make_inner()
inner

Here we configure `OtelModule` with `exporter="none"`. That's an important
detail: the `OtelModule` *does not* manage the global tracer provider when
`exporter="none"` — it only validates config. We've already installed our
in-memory provider in §1, so spans flow straight there.

In [ ]:
reset_sdk()  # Reset the OtelModule's "I already configured the SDK" guard
reset_exporter()

otel_mod = OtelModule({"exporter": "none"}, inner)
type(otel_mod).__name__, otel_mod._inner.name, otel_mod._inner.model_name

In [ ]:
import asyncio


async def run_once(provider, prompt: str = "What is 2+2?"):
    return await provider.invoke([Message(role="user", content=prompt)])


resp = await run_once(otel_mod)
print("response.content:", resp.content)
print("usage:", resp.usage.model_dump())
print()
print_tree(EXPORTER.get_finished_spans())

One span: `arcllm.invoke`. That's the root for *every* call into the module
stack. Now let's wrap it with a parent span the way an agent framework would,
and confirm `arcllm.invoke` auto-nests.

In [ ]:
reset_exporter()


async def run_under_parent():
    tracer = otel_trace.get_tracer("agent.framework")
    with tracer.start_as_current_span("agent.task.research"):
        return await otel_mod.invoke([Message(role="user", content="ping")])


await run_under_parent()
print_tree(EXPORTER.get_finished_spans())

`arcllm.invoke` is a child of `agent.task.research` — exactly what you want
in a distributed trace UI: the agent task, with the LLM call nested inside.

To see real *cross-module* nesting, wrap `OtelModule` around a stack that
also produces spans. We'll stub a `SecurityModule`-shaped child manually
using `_span()` to keep this notebook self-contained and avoid pulling in
extra modules' configs.

In [ ]:
class TracingMiddleware(BaseModule):
    """Toy module that emits a child span and a grandchild span."""

    async def invoke(self, messages, tools=None, **kwargs):
        with self._span("middleware.work"):
            with self._span("middleware.work.subtask"):
                pass
            return await self._inner.invoke(messages, tools, **kwargs)


reset_exporter()
stack = OtelModule({"exporter": "none"}, TracingMiddleware({}, make_inner()))
await run_once(stack)
print_tree(EXPORTER.get_finished_spans())

## 4. Standard `gen_ai.*` attributes

The `OtelModule` tags every root span with the OpenTelemetry *GenAI semantic
conventions*. Vendor dashboards (Datadog APM, Grafana Tempo, Honeycomb,
Jaeger) recognise these and render them as first-class LLM call timelines.

| Attribute | Source | Why |
|-----------|--------|-----|
| `gen_ai.system` | `inner.name` | Vendor identification |
| `gen_ai.request.model` | `inner.model_name` | Which model was *requested* |
| `gen_ai.usage.input_tokens` | `response.usage.input_tokens` | Cost / volume |
| `gen_ai.usage.output_tokens` | `response.usage.output_tokens` | Cost / volume |
| `gen_ai.response.model` | `response.model` | Model that *served* the request (may differ from request) |
| `gen_ai.response.finish_reasons` | `[response.stop_reason]` | Stop reason as a list (per spec) |

Let's read them off the captured span.

In [ ]:
reset_sdk()
reset_exporter()

inner = make_inner(name="anthropic", model="claude-sonnet-4-6")
inner.invoke = AsyncMock(
    return_value=LLMResponse(
        content="42",
        usage=Usage(input_tokens=120, output_tokens=11, total_tokens=131),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    )
)

mod = OtelModule({"exporter": "none"}, inner)
await run_once(mod)

(span,) = EXPORTER.get_finished_spans()
for k, v in span.attributes.items():
    print(f"{k:40s} {v!r}")

Notice `gen_ai.response.finish_reasons` is a *list* with one element. The
spec models it as a list because some providers return multiple completions
per request. ArcLLM normalises to a single response, so the list always has
length 1.

In [ ]:
# Confirm the type — backends rely on this being a sequence.
finish = span.attributes["gen_ai.response.finish_reasons"]
print(type(finish).__name__, finish, "len:", len(finish))

## 5. Queue / wait attributes (new in v0.4)

`QueueModule` (added in v0.4 as part of the Otel → **Queue** → Telemetry stack
reorder) tags the active span with concurrency telemetry. This makes
queue-induced latency visible in the same trace as the actual provider call.

| Attribute | Meaning | Set when |
|-----------|---------|----------|
| `arc.queue.wait_ms` | How long this caller spent in the FIFO before acquiring the semaphore | On every successful enqueue |
| `arc.queue.depth` | Number of waiters when this caller arrived | On every successful enqueue |
| `arc.queue.call_timeout_ms` | The configured send-time timeout (post-acquire) | On every successful enqueue |
| `arc.queue.rejected` | `True` when the call was rejected because `waiters >= max_queued` | On `QueueFullError` only |

Read the source — it's set on the *current* span, which is the
`arcllm.invoke` span when QueueModule sits inside OtelModule:

In [ ]:
import inspect

from arcllm.modules.queue import QueueModule

print(inspect.getsource(QueueModule._set_span_attributes).strip())
print()
print(inspect.getsource(QueueModule._set_rejected_span_attribute).strip())

Let's drive a small concurrency burst through `OtelModule(QueueModule(adapter))`
and read back the per-call queue attributes. We use `max_concurrent=1` so the
second call has to wait.

In [ ]:
import time


class SlowAdapter:
    """Fake adapter with controllable latency, conforming to LLMProvider."""

    name = "anthropic"
    model_name = "claude-sonnet-4-6"

    def __init__(self, delay_s: float = 0.05):
        self._delay = delay_s

    async def invoke(self, messages, tools=None, **kwargs):
        await asyncio.sleep(self._delay)
        return LLMResponse(
            content="ok",
            usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
            model=self.model_name,
            stop_reason="end_turn",
        )

    def validate_config(self) -> bool:
        return True

    async def close(self) -> None:
        return None


reset_sdk()
reset_exporter()

queue_stack = OtelModule(
    {"exporter": "none"},
    QueueModule(
        {"max_concurrent": 1, "max_queued": 4, "call_timeout": 5.0},
        SlowAdapter(delay_s=0.05),
    ),
)


async def burst(n: int):
    return await asyncio.gather(*(run_once(queue_stack, f"q{i}") for i in range(n)))


t0 = time.monotonic()
await burst(3)
print(f"3 calls @ max_concurrent=1 took {time.monotonic() - t0:.2f}s")

In [ ]:
spans = EXPORTER.get_finished_spans()
print(f"captured {len(spans)} spans:\n")
for s in sorted(spans, key=lambda x: x.start_time):
    queue_attrs = {k: v for k, v in s.attributes.items() if k.startswith("arc.queue.")}
    print(s.name, queue_attrs)

The first call waits 0 ms, the next two each wait roughly the previous
call's service time. `arc.queue.depth` shows how many other callers were
ahead. `arc.queue.call_timeout_ms` is constant (configured).

### `arc.queue.rejected` — backpressure

When `waiters >= max_queued`, the queue rejects the call **before** acquiring
the semaphore. The current span (created by the parent `OtelModule`) is
tagged `arc.queue.rejected=True` and the call raises `QueueFullError`.

We force the rejection branch by setting `max_queued=0` and firing two
concurrent calls.

In [ ]:
from arcllm.exceptions import QueueFullError

reset_sdk()
reset_exporter()

# max_queued=0 means: while one call holds the semaphore, the next is
# rejected immediately rather than waiting.
strict = OtelModule(
    {"exporter": "none"},
    QueueModule(
        {"max_concurrent": 1, "max_queued": 0, "call_timeout": 5.0},
        SlowAdapter(delay_s=0.05),
    ),
)


async def burst_strict():
    results = await asyncio.gather(
        run_once(strict, "first"),
        run_once(strict, "second"),
        return_exceptions=True,
    )
    return results


outcomes = await burst_strict()
for r in outcomes:
    print(type(r).__name__, r)

In [ ]:
# The rejected call's span carries arc.queue.rejected=True.
spans = EXPORTER.get_finished_spans()
for s in sorted(spans, key=lambda x: x.start_time):
    rejected = s.attributes.get("arc.queue.rejected", False)
    wait_ms = s.attributes.get("arc.queue.wait_ms", "—")
    print(f"{s.name:18s} rejected={rejected!s:5s} wait_ms={wait_ms}")

## 6. Exporters: in-memory and OTLP

Three exporter values are valid:

| `exporter` | What it does | Suitable for |
|------------|--------------|--------------|
| `"none"` | Skips SDK setup entirely; tracer-provider is whatever you've set up externally | Tests, notebooks, custom setups |
| `"console"` | `ConsoleSpanExporter` — prints span JSON to stdout | Local development |
| `"otlp"` | `OTLPSpanExporter` (gRPC or HTTP) shipped to `endpoint` | Production |

For tests, the canonical pattern is:

1. Install your *own* `TracerProvider` with `InMemorySpanExporter` (we did
   this in §1).
2. Construct `OtelModule({"exporter": "none"}, ...)`. The module skips
   `_setup_sdk()` and your provider stays in place.
3. Read `EXPORTER.get_finished_spans()` and assert.

In [ ]:
from arcllm.exceptions import ArcLLMConfigError

inner = make_inner()

# Valid exporters
for exporter_value in ("otlp", "console", "none"):
    try:
        OtelModule({"exporter": exporter_value}, inner) if exporter_value == "none" else None
        print(f"valid: exporter={exporter_value!r}")
    except ArcLLMConfigError as exc:
        print(f"invalid: exporter={exporter_value!r} ({exc})")

# Invalid exporter rejected at construction
try:
    OtelModule({"exporter": "prometheus"}, inner)
except ArcLLMConfigError as exc:
    print("rejected: exporter='prometheus' →", exc)

### OTLP configuration (config-only — no real network)

We don't actually ship spans to a collector here, but we can inspect the
config keys the module accepts and the TLS-builder helper. This is the
contract a production deployment fills in via TOML.

In [ ]:
from arcllm.modules.otel import _VALID_CONFIG_KEYS, _build_tls_kwargs

print("OtelModule valid config keys:")
for k in sorted(_VALID_CONFIG_KEYS):
    print(f"  {k}")
print()

# TLS builder pulls cert/key paths out of config; safe to call with strings.
tls = _build_tls_kwargs(
    {
        "certificate_file": "/etc/ssl/ca.pem",
        "client_key_file": "/etc/ssl/client.key",
        "client_cert_file": "/etc/ssl/client.crt",
    }
)
print("TLS kwargs for OTLP exporter:")
for k, v in tls.items():
    print(f"  {k}: {v}")

### Sample TOML for OTLP

```toml
[modules.otel]
enabled = true
exporter = "otlp"
endpoint = "https://otel.internal:4317"
protocol = "grpc"
service_name = "arc-prod-agents"
sample_rate = 0.1                    # 10% sampling for high-throughput pools
headers = { authorization = "Bearer secret-from-vault" }
insecure = false
certificate_file = "/etc/arc/tls/ca.pem"
client_key_file  = "/etc/arc/tls/client.key"
client_cert_file = "/etc/arc/tls/client.crt"
timeout_ms = 10000
max_batch_size = 512
max_queue_size = 2048
schedule_delay_ms = 5000

[modules.otel.resource_attributes]
deployment.environment = "production"
service.version = "1.0.0"
```

Sample-rate validation rejects out-of-range values:

In [ ]:
for sr in (-0.1, 0.0, 0.5, 1.0, 1.5):
    try:
        OtelModule({"exporter": "none", "sample_rate": sr}, inner)
        print(f"sample_rate={sr}: ok")
    except ArcLLMConfigError as exc:
        print(f"sample_rate={sr}: rejected — {exc}")

In [ ]:
# Unknown config keys are rejected (typo defence)
try:
    OtelModule({"exportr": "otlp"}, inner)  # 'exportr' (typo)
except ArcLLMConfigError as exc:
    print("rejected:", exc)

## 7. Resource attributes

`Resource` attributes describe **where** the spans came from — the service,
deployment, host, version. Unlike span attributes (per-call), resource
attributes apply to every span emitted by the provider.

`OtelModule._setup_sdk()` builds the resource from two sources:

1. `service.name` from the `service_name` config key (default `"arcllm"`).
2. Anything you put under `resource_attributes` is merged on top.

We can construct the resource the same way `_setup_sdk` does, without
touching the global provider:

In [ ]:
from opentelemetry.sdk.resources import Resource

config = {
    "service_name": "arc-research-agents",
    "resource_attributes": {
        "deployment.environment": "production",
        "service.version": "1.0.0",
        "deployment.region": "us-gov-west-1",
    },
}

resource_attrs: dict = {"service.name": config["service_name"]}
resource_attrs.update(config["resource_attributes"])
resource = Resource.create(resource_attrs)

for k, v in resource.attributes.items():
    if k.startswith(("service.", "deployment.")):
        print(f"{k:32s} {v}")

In a Datadog APM dashboard you'd be able to filter by
`deployment.environment=production` and `service.name=arc-research-agents`,
seeing spans from every agent in that pool grouped together. In a Grafana
Tempo trace you can group by `service.version` to see whether a recent
deploy changed your latency profile.

## 8. SDK reset with `clear_cache()`

`OtelModule` keeps a module-level `_sdk_configured` flag so it never
reconfigures the SDK twice during a single process. That's correct in
production — but in tests, notebooks, and any session that re-loads
config, you need to be able to reset it.

Two reset paths exist:

1. **Direct**: `arcllm.modules.otel.reset_sdk()` — flips the flag back to
   `False`.
2. **Indirect via registry**: `arcllm.registry.clear_cache()` calls
   `reset_sdk()` along with clearing every other registry cache (provider
   configs, adapter classes, rate-limit buckets, telemetry budgets).

Both are safe to call repeatedly. They reset the *flag* but do not tear
down the global tracer provider — that's intentional, because tearing it
down would break any consumer that grabbed a tracer reference earlier.

In [ ]:
from arcllm import modules as _modules_pkg
from arcllm.modules import otel as _otel_mod

print("reset_sdk:", _otel_mod.reset_sdk)
print()
print(inspect.getsource(_otel_mod.reset_sdk).strip())

In [ ]:
# Demonstrate the flag flips back and forth.
_otel_mod._sdk_configured = True
print("before reset:", _otel_mod._sdk_configured)

_otel_mod.reset_sdk()
print("after reset_sdk():", _otel_mod._sdk_configured)

`clear_cache()` chains into `reset_sdk()` so a single call covers every
arcllm reset path — provider configs, adapter classes, rate-limit token
buckets, telemetry budgets, telemetry global defaults, and the OTel SDK
flag.

In [ ]:
from arcllm import registry as _registry_mod

src = inspect.getsource(_registry_mod.clear_cache)
print(src.strip())

In [ ]:
# Round-trip: flip the flag, then clear via the registry.
_otel_mod._sdk_configured = True
print("before clear_cache:", _otel_mod._sdk_configured)

_registry_mod.clear_cache()
print("after clear_cache(): ", _otel_mod._sdk_configured)

### Idempotence: calling `_setup_sdk()` twice is a no-op

If `_sdk_configured` is already `True`, `_setup_sdk()` returns immediately.
That keeps multiple `OtelModule` instances in the same process from racing
to install conflicting tracer providers.

In [ ]:
src = inspect.getsource(_otel_mod._setup_sdk)
# Print just the guard so the cell stays small.
for line in src.splitlines()[:6]:
    print(line)

## 9. Summary

Two-layer instrumentation: `OtelModule` builds the *root* `arcllm.invoke`
span, and `BaseModule._span()` lets every other module emit nested children
without OTel boilerplate. Together they produce a clean trace tree that
slots into agent-framework parent spans automatically.

What was demonstrated end-to-end:

- **Setup with `InMemorySpanExporter`** so every cell ran offline.
- **Span hierarchy**: root `arcllm.invoke` plus child spans from custom
  middleware; auto-nesting under an agent-framework parent.
- **Standard `gen_ai.*` attributes**: system, request/response model, input
  / output tokens, finish reasons (always a list).
- **`arc.queue.*` attributes (v0.4 new)**: `wait_ms`, `depth`,
  `call_timeout_ms` on every successful enqueue, `rejected=True` on
  `QueueFullError`.
- **Exporter validation**: `"otlp" | "console" | "none"` — typos are
  rejected at construction. OTLP TOML config plus TLS kwargs builder shown
  without making real network calls.
- **Resource attributes**: how `service.name` and `resource_attributes` are
  merged for backend filtering and grouping.
- **`clear_cache()` ↔ `reset_sdk()`**: idempotent flag-based reset that
  keeps tests deterministic without tearing down the tracer provider.

Public API covered:

| Symbol | Where |
|--------|-------|
| `OtelModule` | `arcllm.modules.otel` |
| `OtelModule._VALID_CONFIG_KEYS` | `arcllm.modules.otel` |
| `OtelModule._build_tls_kwargs` | `arcllm.modules.otel` |
| `reset_sdk` | `arcllm.modules.otel` |
| `BaseModule._tracer`, `BaseModule._span` | `arcllm.modules.base` |
| `QueueModule._set_span_attributes` (`arc.queue.*`) | `arcllm.modules.queue` |
| `QueueModule._set_rejected_span_attribute` (`arc.queue.rejected`) | `arcllm.modules.queue` |
| `clear_cache` | `arcllm.registry` |
| `ArcLLMConfigError`, `QueueFullError` | `arcllm.exceptions` |

Next up: `12-security-layer.ipynb` shows how the SecurityModule layer
contributes its own `security.*` child spans on top of everything you saw
here.